# OTTO Multi-Task Recommender — Complete Notebook

Run this end-to-end on Kaggle (free T4 GPU). Phases:
1. Install dependencies
2. Explore the OTTO data
3. Build sequences
4. Train SASRec
5. FAISS retrieval + Recall@20
6. ONNX export + latency benchmark

In [ ]:
!pip install -q faiss-cpu onnx onnxruntime onnxscript tqdm
# onnxscript is required by torch.onnx.export in PyTorch 2.x

In [ ]:
# ─── Imports & device setup ───────────────────────────────────────────────────
import sys, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm
import faiss
import onnx, onnxruntime as ort
from collections import defaultdict, Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ─── Phase 1: Load & explore data ────────────────────────────────────────────
TRAIN_PATH = '/kaggle/input/otto-recommender-system/train.jsonl'

# Full dataset is ~12M sessions / 30 GB RAM — too large for Kaggle's free tier.
# 500_000 sessions uses ~2 GB RAM and trains in ~15 min on T4.
# Raise to 1_000_000 if you have a Kaggle Pro account (30 GB RAM).
MAX_SESSIONS = 500_000

sessions = []
with open(TRAIN_PATH) as f:
    for line in f:
        sessions.append(json.loads(line))
        if len(sessions) >= MAX_SESSIONS:
            break

print(f'Sessions loaded: {len(sessions):,}')
lengths = [len(s['events']) for s in sessions]
print(f'Avg session length: {np.mean(lengths):.1f}  |  Max: {max(lengths)}')
counts = Counter(e['type'] for s in sessions for e in s['events'])
print('Event distribution:', dict(counts))

In [ ]:
# ─── Phase 2: Build vocabulary ────────────────────────────────────────────────
PAD_TOKEN   = 0
ITEM_OFFSET = 2
MAX_LEN     = 50
EVENT_TYPE_MAP = {'clicks': 0, 'carts': 1, 'orders': 2}

unique_aids  = sorted(set(e['aid'] for s in sessions for e in s['events']))
aid_to_token = {aid: idx + ITEM_OFFSET for idx, aid in enumerate(unique_aids)}
token_to_aid = {v: k for k, v in aid_to_token.items()}
VOCAB_SIZE   = len(aid_to_token) + ITEM_OFFSET
print(f'Vocabulary size: {VOCAB_SIZE:,}')

with open('vocab.json', 'w') as f:
    json.dump({'aid_to_token': {str(k):v for k,v in aid_to_token.items()},
               'token_to_aid': {str(k):v for k,v in token_to_aid.items()}}, f)

# Pre-tokenise sessions into compact int lists — no sliding window yet.
# A list of N ints uses ~56 + 8N bytes vs a dict of MAX_LEN ints = ~1400 bytes.
# For 500k sessions this saves ~4 GB of RAM compared to pre-building all samples.
tokenised = []   # list of (token_ids, type_ids) per session
for s in tqdm(sessions, desc='Tokenising'):
    evts = s['events']
    if len(evts) < 2: continue
    tokenised.append((
        [aid_to_token[e['aid']] for e in evts],
        [EVENT_TYPE_MAP[e['type']] for e in evts],
    ))

# Build a flat index: (session_idx, position) for every valid (input, target) pair.
# This is just a list of two small ints per sample — negligible RAM.
index = []
for si, (toks, _) in enumerate(tokenised):
    for pos in range(len(toks) - 1):   # pos = index of last input token
        index.append((si, pos))

print(f'Total samples (index entries): {len(index):,}')

In [ ]:
# ─── Dataset & DataLoader ─────────────────────────────────────────────────────
class OTTODataset(Dataset):
    """
    Lazy dataset — stores only the flat index (session_idx, position).
    Sequences are built on-the-fly in __getitem__, so RAM usage is O(sessions)
    rather than O(total sliding-window samples).
    """
    def __init__(self, index, tokenised, max_len):
        self.index     = index       # list of (session_idx, position)
        self.tokenised = tokenised   # list of (token_ids, type_ids)
        self.max_len   = max_len

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        si, pos      = self.index[idx]
        token_ids, type_ids = self.tokenised[si]

        # Build the input sequence ending at position `pos`
        start      = max(0, pos - self.max_len + 1)
        seq        = token_ids[start : pos + 1]
        pad_len    = self.max_len - len(seq)
        seq_padded = [0] * pad_len + seq

        return {
            'input_ids':   torch.tensor(seq_padded,        dtype=torch.long),
            'target_id':   torch.tensor(token_ids[pos+1],  dtype=torch.long),
            'target_type': torch.tensor(type_ids[pos+1],   dtype=torch.long),
        }

# Split index into train / val
val_n     = int(len(index) * 0.05)
train_idx = index[:-val_n]
val_idx   = index[-val_n:]

train_ds = OTTODataset(train_idx, tokenised, MAX_LEN)
val_ds   = OTTODataset(val_idx,   tokenised, MAX_LEN)

BATCH_SIZE   = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

In [ ]:
# ─── Phase 3: Define SASRec model (inline for notebook convenience) ───────────
class SASRecBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(4*d_model, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        a, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.drop1(a))
        return self.norm2(x + self.drop2(self.ffn(x)))

class SASRec(nn.Module):
    def __init__(self, vocab_size, d_model=128, max_len=50, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.item_emb     = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb      = nn.Embedding(max_len, d_model)
        self.emb_dropout  = nn.Dropout(dropout)
        self.blocks       = nn.ModuleList([SASRecBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head_click   = nn.Linear(d_model, vocab_size, bias=False)
        self.head_cart    = nn.Linear(d_model, vocab_size, bias=False)
        self.head_order   = nn.Linear(d_model, vocab_size, bias=False)
        self.head_click.weight = self.head_cart.weight = self.head_order.weight = self.item_emb.weight
        self.max_len      = max_len
        for m in self.modules():
            if isinstance(m, nn.Linear): nn.init.xavier_uniform_(m.weight)
            elif isinstance(m, nn.Embedding): nn.init.normal_(m.weight, std=0.02)

    def encode(self, x):
        b, s = x.shape
        pos  = torch.arange(s, device=x.device).unsqueeze(0)
        h    = self.emb_dropout(self.item_emb(x) + self.pos_emb(pos))
        mask = torch.triu(torch.ones(s, s, device=x.device, dtype=torch.bool), diagonal=1)
        for blk in self.blocks: h = blk(h, mask)
        return h

    def forward(self, x):
        h = self.encode(x)[:, -1, :]
        return {'logits_click': self.head_click(h), 'logits_cart': self.head_cart(h), 'logits_order': self.head_order(h)}

    def get_item_embeddings(self):
        return self.item_emb.weight.detach()

model = SASRec(VOCAB_SIZE).to(DEVICE)

# Use both GPUs if available
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}')
    model = nn.DataParallel(model)
else:
    print(f'Using single GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

# Helper to access the underlying model regardless of DataParallel wrapping
raw_model = model.module if isinstance(model, nn.DataParallel) else model

print(f'Parameters: {sum(p.numel() for p in raw_model.parameters() if p.requires_grad):,}')

In [ ]:
# ─── Train ────────────────────────────────────────────────────────────────────
# ROOT CAUSE of OOM: CrossEntropyLoss over full vocab.
# With vocab_size ~200k and batch 64, logits tensor = 64×200k×3 heads = ~150 MB
# per step — fine. But at batch 512 it was 64×bigger → OOM.
#
# BETTER FIX: sampled softmax. Instead of scoring all 200k items,
# we score: 1 positive (true next item) + N_NEG random negatives per sample.
# This is standard practice in large-vocab recommenders (YouTube DNN, PinSage).
# Memory: O(batch × N_NEG) instead of O(batch × vocab_size).

N_NEG = 200

def sampled_softmax_loss(h, target_id, item_emb_weight, n_neg):
    batch      = h.size(0)
    vocab_size = item_emb_weight.size(0)
    pos_emb    = item_emb_weight[target_id]
    pos_score  = (h * pos_emb).sum(dim=-1)
    neg_ids    = torch.randint(2, vocab_size, (batch, n_neg), device=h.device)
    neg_emb    = item_emb_weight[neg_ids]
    neg_scores = torch.bmm(neg_emb, h.unsqueeze(-1)).squeeze(-1)
    diff       = neg_scores - pos_score.unsqueeze(1)
    return torch.nn.functional.softplus(diff).mean()

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

W_CLICK, W_CART, W_ORDER = 1.0, 2.0, 4.0
EPOCHS   = 10
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.
    for batch in tqdm(train_loader, desc=f'Epoch {epoch} train'):
        ids = batch['input_ids'].to(DEVICE)
        tgt = batch['target_id'].to(DEVICE)

        # raw_model.encode works correctly whether wrapped in DataParallel or not.
        # DataParallel.forward() splits the batch across GPUs automatically,
        # but encode() and item_emb must be called on the underlying module.
        h = raw_model.encode(ids)[:, -1, :]   # (batch, d)
        W = raw_model.item_emb.weight          # (vocab, d)

        l_click = sampled_softmax_loss(h, tgt, W, N_NEG)
        l_cart  = sampled_softmax_loss(h, tgt, W, N_NEG)
        l_order = sampled_softmax_loss(h, tgt, W, N_NEG)
        loss    = W_CLICK*l_click + W_CART*l_cart + W_ORDER*l_order

        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval(); val_loss = 0.
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(DEVICE)
            tgt = batch['target_id'].to(DEVICE)
            h   = raw_model.encode(ids)[:, -1, :]
            W   = raw_model.item_emb.weight
            l_click = sampled_softmax_loss(h, tgt, W, N_NEG)
            l_cart  = sampled_softmax_loss(h, tgt, W, N_NEG)
            l_order = sampled_softmax_loss(h, tgt, W, N_NEG)
            val_loss += (W_CLICK*l_click + W_CART*l_cart + W_ORDER*l_order).item()

    scheduler.step()
    tl = train_loss/len(train_loader); vl = val_loss/len(val_loader)
    print(f'Epoch {epoch}: train={tl:.4f}  val={vl:.4f}')
    if vl < best_val:
        best_val = vl
        torch.save({'model_state': raw_model.state_dict(), 'config': {'d_model':128,'max_len':MAX_LEN,'n_heads':4,'n_layers':2}, 'vocab_size': VOCAB_SIZE, 'aid_to_token': aid_to_token}, 'best_model.pt')
        print('  Saved.')

In [ ]:
# ─── Phase 4: FAISS retrieval + Recall@20 ────────────────────────────────────
raw_model.eval()
item_embs = raw_model.get_item_embeddings().cpu().numpy().astype(np.float32)
faiss.normalize_L2(item_embs)
index = faiss.IndexFlatIP(item_embs.shape[1])
index.add(item_embs)
print(f'FAISS index: {index.ntotal:,} vectors')

hits, total = 0, 0
val_loader2 = DataLoader(val_ds, batch_size=256, shuffle=False)
with torch.no_grad():
    for batch in tqdm(val_loader2, desc='Recall@20'):
        ids  = batch['input_ids'].to(DEVICE)
        tgts = batch['target_id'].numpy()
        h    = raw_model.encode(ids)[:, -1, :].cpu().numpy().astype(np.float32)
        faiss.normalize_L2(h)
        _, top20 = index.search(h, 20)
        for i, tgt in enumerate(tgts):
            if tgt in top20[i]: hits += 1
        total += len(tgts)

print(f'Recall@20: {hits/total:.4f}')

In [ ]:
# ─── Phase 5: ONNX export + benchmark ────────────────────────────────────────
# Move model to CPU for export — ONNX tracing works on CPU
raw_model.cpu().eval()
dummy = torch.zeros(1, MAX_LEN, dtype=torch.long)

with torch.no_grad():
    torch.onnx.export(
        raw_model, dummy, 'sasrec.onnx',
        opset_version=14,          # 14 is stable across all torch 2.x + onnxruntime versions
        input_names=['input_ids'],
        output_names=['logits_click','logits_cart','logits_order'],
        dynamic_axes={'input_ids':{0:'batch'},'logits_click':{0:'batch'},'logits_cart':{0:'batch'},'logits_order':{0:'batch'}},
        do_constant_folding=True,
    )
print('Exported to sasrec.onnx')

# Benchmark
sess = ort.InferenceSession('sasrec.onnx', providers=['CPUExecutionProvider'])
dummy_np = np.zeros((1, MAX_LEN), dtype=np.int64)

for _ in range(20): sess.run(None, {'input_ids': dummy_np})

N = 200
t0 = time.perf_counter()
for _ in range(N): sess.run(None, {'input_ids': dummy_np})
ort_ms = (time.perf_counter()-t0)*1000/N

dummy_cpu = torch.zeros(1, MAX_LEN, dtype=torch.long)
for _ in range(20):
    with torch.no_grad(): raw_model(dummy_cpu)
t0 = time.perf_counter()
for _ in range(N):
    with torch.no_grad(): raw_model(dummy_cpu)
pt_ms = (time.perf_counter()-t0)*1000/N

print(f'\nPyTorch CPU: {pt_ms:.2f} ms/inf')
print(f'ONNX RT CPU: {ort_ms:.2f} ms/inf')
print(f'Speedup:     {pt_ms/ort_ms:.2f}x')

import os
print(f'\nModel sizes:')
print(f'  best_model.pt : {os.path.getsize("best_model.pt")/1e6:.1f} MB')
print(f'  sasrec.onnx   : {os.path.getsize("sasrec.onnx")/1e6:.1f} MB')